# 03. 운영 control band와 SDLC 환류

목표: 모델이 이상을 임의로 판정하지 않도록 결정론적 탐지기를 만들고, 이탈 강도에 따라 권한을 제한한 뒤 새 `intent.md` 데이터로 환류합니다. Python 표준 라이브러리만 사용합니다.

In [ ]:
from dataclasses import dataclass
from statistics import mean, stdev

@dataclass(frozen=True)
class Breach:
    value: float
    baseline_mean: float
    baseline_std: float
    sigma: float
    tier: str

def detect(baseline, current) -> Breach:
    center = mean(baseline)
    spread = stdev(baseline)
    sigma = 0.0 if spread == 0 else abs(current - center) / spread
    if sigma >= 3:
        tier = 'propose'
    elif sigma >= 2:
        tier = 'diagnose'
    elif sigma >= 1:
        tier = 'log'
    else:
        tier = 'normal'
    return Breach(current, center, spread, sigma, tier)

baseline_5xx = [0.009, 0.011, 0.010, 0.012, 0.008, 0.010, 0.009]
breach = detect(baseline_5xx, current=0.025)
print(breach)
assert breach.tier == 'propose'


## 위험 단계가 도구를 제한한다

탐지 결과는 모델이 사용할 수 있는 행동의 상한을 결정합니다. 가장 높은 단계도 임의의 운영 셸이 아니라 PR과 사전 승인 런북만 허용합니다.

In [ ]:
TOOLS_BY_TIER = {
    'normal': set(),
    'log': {'write_observation'},
    'diagnose': {'read_logs', 'read_metrics', 'read_code'},
    'propose': {'read_logs', 'read_metrics', 'read_code', 'open_pull_request', 'run_preapproved_rollback'},
}

def authorize(tier, requested):
    allowed = TOOLS_BY_TIER[tier]
    return requested in allowed

for action in ('read_logs', 'open_pull_request', 'production_shell'):
    print(action, authorize(breach.tier, action))
assert not authorize(breach.tier, 'production_shell')


## 진단을 새 의도로 변환

아래 함수는 파일을 쓰지 않고 `intent.md`에 들어갈 구조화 데이터를 만듭니다. 실제 시스템은 관찰 원본 링크, 배포 SHA, 승인자, 개인정보 제거 여부도 기록해야 합니다.

In [ ]:
def incident_intent(metric, breach, deployment_sha):
    return {
        'problem': f'{metric}가 기준선에서 {breach.sigma:.2f}σ 이탈',
        'evidence': {
            'current': breach.value,
            'baseline_mean': round(breach.baseline_mean, 5),
            'baseline_std': round(breach.baseline_std, 5),
            'deployment_sha': deployment_sha,
        },
        'outcome': '원인 규명과 오류율 정상 범위 복귀',
        'affected_systems': ['public-api'],
        'constraints': ['운영 직접 수정 금지', '모든 변경은 PR 게이트 통과'],
        'open_questions': ['특정 endpoint에 집중되는가?', '최근 배포와 시간상 연관되는가?'],
        'response_tier': breach.tier,
    }

intent = incident_intent('post_deploy_5xx_rate', breach, 'abc1234')
for key, value in intent.items():
    print(f'{key}: {value}')
assert intent['response_tier'] == 'propose'
assert '운영 직접 수정 금지' in intent['constraints']


## 확장 과제

1. 단일 점 이탈뿐 아니라 연속 상승을 감지하는 Western Electric 규칙을 추가하세요.
2. 오탐을 기각하면 기준선을 즉시 바꾸지 말고 검토 대기열에 넣으세요.
3. 해결된 사고를 eval 사례로 바꾸는 함수를 작성하세요.
4. 롤백은 현재 배포가 사전 승인 목록에 있을 때만 허용하도록 강화하세요.